In [5]:

import numpy as np 
import pandas as pd 


In [6]:
results = pd.read_csv("https://raw.githubusercontent.com/martj42/international_results/master/results.csv")
results['date'] = pd.to_datetime(results['date'])
results = results.sort_values('date').reset_index(drop=True)

In [7]:
ratings = {}
def get_ratings(team):
    return ratings.get(team,1500)

print(get_ratings("Pakistan"))  # should print 1500

def expected_score(rating_a, rating_b):
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

print(expected_score(1500, 1500))  # should print 0.5 (equal teams, 50/50)
print(expected_score(1900, 1500))  # should print something high, like 0.9+

1500
0.5
0.9090909090909091


In [8]:
def update_rating(rating_a, rating_b, score_a, k=20):
    expected_a = expected_score(rating_a, rating_b)
    new_rating_a = rating_a + k * (score_a - expected_a)
    return new_rating_a

new_a = update_rating(1500, 1500, score_a=1)
print(new_a)  # A won when it was only "expected" to win half the time -> rating should go UP

1510.0


In [9]:
new_a = update_rating(1500, 1900, score_a=1)
print(new_a)

1518.1818181818182


In [10]:
def play_match(rating_a, rating_b, score_a):
    expected_a = expected_score(rating_a, rating_b)
    expected_b = 1 - expected_a          # if A has 70% chance, B has 30%
    score_b = 1 - score_a                # if A won, B lost, etc.

    new_a = rating_a + 20 * (score_a - expected_a)
    new_b = rating_b + 20 * (score_b - expected_b)
    return new_a, new_b

new_a, new_b = play_match(1500, 1900, score_a=1)
print(new_a, new_b)

1518.1818181818182 1881.8181818181818


In [11]:
# --- Setup ---
results = results.dropna(subset=['home_team', 'away_team'])

ratings = {}
history = []

def get_rating(team):
    return ratings.get(team, 1500)

def expected_score(rating_a, rating_b):
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

def play_match(rating_a, rating_b, score_a):
    expected_a = expected_score(rating_a, rating_b)
    expected_b = 1 - expected_a
    score_b = 1 - score_a

    new_a = rating_a + 20 * (score_a - expected_a)
    new_b = rating_b + 20 * (score_b - expected_b)
    return new_a, new_b

# --- Main Elo loop ---
for row in results.itertuples():
    home = row.home_team
    away = row.away_team

    if row.home_score > row.away_score:
        score_home = 1
    elif row.home_score < row.away_score:
        score_home = 0
    else:
        score_home = 0.5

    rating_home = get_rating(home)
    rating_away = get_rating(away)

    # save snapshot BEFORE this match updates anything
    history.append({'date': row.date, 'team': home, 'elo_before': rating_home})
    history.append({'date': row.date, 'team': away, 'elo_before': rating_away})

    new_home, new_away = play_match(rating_home, rating_away, score_home)

    ratings[home] = new_home
    ratings[away] = new_away

# --- Build the lookup table (runs once, after the loop) ---
elo_history = pd.DataFrame(history)

# --- Sanity checks ---
top_10 = sorted(ratings.items(), key=lambda x: x[1], reverse=True)[:10]
for team, rating in top_10:
    print(team, round(rating, 1))

Argentina 2008.5
Spain 1999.6
France 1971.0
England 1926.3
Brazil 1917.9
Portugal 1900.4
Colombia 1894.2
Netherlands 1881.7
Germany 1879.7
Morocco 1862.6


In [12]:
top_10 = sorted(ratings.items(), key=lambda x: x[1], reverse=True)[:10]
for team, rating in top_10:
    print(team, round(rating, 1))

elo_history.head(10)

Argentina 2008.5
Spain 1999.6
France 1971.0
England 1926.3
Brazil 1917.9
Portugal 1900.4
Colombia 1894.2
Netherlands 1881.7
Germany 1879.7
Morocco 1862.6


,date,team,elo_before
0,1872-11-30,Scotland,1500.000000
1,1872-11-30,England,1500.000000
2,1873-03-08,England,1500.000000
3,1873-03-08,Scotland,1500.000000
4,1874-03-07,Scotland,1490.000000
5,1874-03-07,England,1510.000000
6,1875-03-06,England,1499.424989
7,1875-03-06,Scotland,1500.575011
8,1876-03-04,Scotland,1500.541911
9,1876-03-04,England,1499.458089


In [13]:
# Merge elo_before onto results for home team
results = results.merge(
    elo_history.rename(columns={'team': 'home_team', 'elo_before': 'home_elo'}),
    on=['date', 'home_team'],
    how='left'
)

# Merge elo_before onto results for away team
results = results.merge(
    elo_history.rename(columns={'team': 'away_team', 'elo_before': 'away_elo'}),
    on=['date', 'away_team'],
    how='left'
)

# Build the core feature
results['elo_diff'] = results['home_elo'] - results['away_elo']

# Target variable: did the home team win?
results['home_win'] = (results['home_score'] > results['away_score']).astype(int)

results[['date', 'home_team', 'away_team', 'home_elo', 'away_elo', 'elo_diff', 'home_win']].head(10)

,date,home_team,away_team,home_elo,away_elo,elo_diff,home_win
0,1872-11-30,Scotland,England,1500.000000,1500.000000,0.000000,0
1,1873-03-08,England,Scotland,1500.000000,1500.000000,0.000000,1
2,1874-03-07,Scotland,England,1490.000000,1510.000000,-20.000000,1
3,1875-03-06,England,Scotland,1499.424989,1500.575011,-1.150023,0
4,1876-03-04,Scotland,England,1500.541911,1499.458089,1.083822,1
5,1876-03-25,Scotland,Wales,1510.510716,1500.000000,10.510716,1
6,1877-03-03,England,Scotland,1489.489284,1520.208286,-30.719002,0
7,1877-03-05,Wales,Scotland,1490.302430,1529.326419,-39.023988,0
8,1878-03-02,Scotland,England,1538.207918,1480.371151,57.836767,1
9,1878-03-23,Scotland,Wales,1546.558450,1481.420932,65.137518,1


In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss

# drop rows with missing elo (first-ever matches for brand new teams have no prior elo... actually these are fine, just double-check no NaNs slipped in)
model_data = results.dropna(subset=['elo_diff', 'home_win'])

X = model_data[['elo_diff']]
y = model_data['home_win']

# chronological split: train on older matches, test on more recent ones
split_index = int(len(model_data) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

model = LogisticRegression()
model.fit(X_train, y_train)

preds = model.predict(X_test)
probs = model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, preds))
print("Log Loss:", log_loss(y_test, probs))

Accuracy: 0.7017051153460381
Log Loss: 0.5666998167329093


In [15]:
# --- 1. Neutral venue feature ---
results['neutral_venue'] = results['neutral'].astype(int)  # True/False -> 1/0

# --- 2. Recent form (rolling goals for/against, last 5 matches, leakage-safe) ---
home_long = results[['date', 'home_team', 'home_score', 'away_score']].rename(
    columns={'home_team': 'team', 'home_score': 'goals_for', 'away_score': 'goals_against'})
away_long = results[['date', 'away_team', 'away_score', 'home_score']].rename(
    columns={'away_team': 'team', 'away_score': 'goals_for', 'home_score': 'goals_against'})

team_matches = pd.concat([home_long, away_long]).sort_values(['team', 'date']).reset_index(drop=True)

# shift(1) ensures we only use matches BEFORE the current one -- no leakage
team_matches['form_gf'] = team_matches.groupby('team')['goals_for'] \
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
team_matches['form_ga'] = team_matches.groupby('team')['goals_against'] \
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())

# teams with no prior matches get filled with the global average
team_matches['form_gf'] = team_matches['form_gf'].fillna(team_matches['form_gf'].mean())
team_matches['form_ga'] = team_matches['form_ga'].fillna(team_matches['form_ga'].mean())

form_lookup = team_matches[['date', 'team', 'form_gf', 'form_ga']]

# --- 3. Merge form onto results (home + away separately) ---
results = results.merge(
    form_lookup.rename(columns={'team': 'home_team', 'form_gf': 'home_form_gf', 'form_ga': 'home_form_ga'}),
    on=['date', 'home_team'], how='left'
)
results = results.merge(
    form_lookup.rename(columns={'team': 'away_team', 'form_gf': 'away_form_gf', 'form_ga': 'away_form_ga'}),
    on=['date', 'away_team'], how='left'
)

# --- 4. Final feature set ---
features = ['elo_diff', 'neutral_venue', 'home_form_gf', 'home_form_ga', 'away_form_gf', 'away_form_ga']
model_data = results.dropna(subset=features + ['home_win'])

X = model_data[features]
y = model_data['home_win']

split_index = int(len(model_data) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

# --- 5. Compare Logistic Regression vs XGBoost, honestly ---
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, log_loss

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)
lr_probs = log_reg.predict_proba(X_test)[:, 1]
lr_preds = log_reg.predict(X_test)

xgb = XGBClassifier(max_depth=4, n_estimators=150, eval_metric='logloss')
xgb.fit(X_train, y_train)
xgb_probs = xgb.predict_proba(X_test)[:, 1]
xgb_preds = xgb.predict(X_test)

print("Logistic Regression -> Accuracy:", accuracy_score(y_test, lr_preds), "| Log Loss:", log_loss(y_test, lr_probs))
print("XGBoost             -> Accuracy:", accuracy_score(y_test, xgb_preds), "| Log Loss:", log_loss(y_test, xgb_probs))

Logistic Regression -> Accuracy: 0.705631729800914 | Log Loss: 0.5650576157770184
XGBoost             -> Accuracy: 0.6989697110543032 | Log Loss: 0.5740784517842409


In [16]:
# Pull out 2022 World Cup knockout matches from your existing results table
backtest = results[
    (results['tournament'] == 'FIFA World Cup') &
    (results['date'].dt.year == 2022)
].dropna(subset=features)

X_backtest = backtest[features]
backtest_probs = log_reg.predict_proba(X_backtest)[:, 1]

backtest_view = backtest[['date', 'home_team', 'away_team', 'home_score', 'away_score']].copy()
backtest_view['model_home_win_prob'] = backtest_probs

# focus on the knockout stage matches specifically (round of 16 onward)
backtest_view.sort_values('date').tail(15)

,date,home_team,away_team,home_score,away_score,model_home_win_prob
60204,2022-12-03,Argentina,Australia,2.0,1.0,0.774675
60205,2022-12-04,France,Poland,3.0,1.0,0.602853
60206,2022-12-04,England,Senegal,3.0,0.0,0.685713
60207,2022-12-05,Japan,Croatia,1.0,1.0,0.338887
60208,2022-12-05,Brazil,South Korea,4.0,1.0,0.763912
60210,2022-12-06,Portugal,Switzerland,6.0,1.0,0.601714
60209,2022-12-06,Morocco,Spain,0.0,0.0,0.299407
60212,2022-12-09,Croatia,Brazil,1.0,1.0,0.220566
60213,2022-12-09,Netherlands,Argentina,2.0,2.0,0.403134
60214,2022-12-10,Morocco,Portugal,1.0,0.0,0.338717


In [17]:
# --- Get latest known form for each team (their most recent match's rolling averages) ---
latest_form = team_matches.sort_values('date').groupby('team').tail(1).set_index('team')

def get_form(team):
    if team in latest_form.index:
        return latest_form.loc[team, 'form_gf'], latest_form.loc[team, 'form_ga']
    else:
        return team_matches['form_gf'].mean(), team_matches['form_ga'].mean()

def predict_match(team_a, team_b, neutral=1):
    elo_a = get_rating(team_a)
    elo_b = get_rating(team_b)
    elo_diff = elo_a - elo_b

    form_gf_a, form_ga_a = get_form(team_a)
    form_gf_b, form_ga_b = get_form(team_b)

    row = pd.DataFrame([{
        'elo_diff': elo_diff,
        'neutral_venue': neutral,
        'home_form_gf': form_gf_a,
        'home_form_ga': form_ga_a,
        'away_form_gf': form_gf_b,
        'away_form_ga': form_ga_b
    }])

    prob_a = log_reg.predict_proba(row[features])[0, 1]
    print(f"{team_a} win probability: {prob_a:.1%}")
    print(f"{team_b} win probability: {1 - prob_a:.1%}")
    return prob_a

# --- Predict the real semifinals ---
print("Semifinal 1:")
predict_match("France", "Spain")

print("\nSemifinal 2:")
predict_match("England", "Argentina")

Semifinal 1:
France win probability: 47.5%
Spain win probability: 52.5%

Semifinal 2:
England win probability: 38.1%
Argentina win probability: 61.9%


np.float64(0.3807576735210193)

In [18]:
!pip install kaleido==0.2.1
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Bar(
    y=['France vs Spain', 'England vs Argentina'],
    x=[47.5, 38.1],
    name='Team A',
    orientation='h',
    marker_color='#4C72B0',
    text=['France 47.5%', 'England 38.1%'],
    textposition='inside'
))

fig.add_trace(go.Bar(
    y=['France vs Spain', 'England vs Argentina'],
    x=[52.5, 61.9],
    name='Team B',
    orientation='h',
    marker_color='#DD8452',
    text=['Spain 52.5%', 'Argentina 61.9%'],
    textposition='inside'
))

fig.update_layout(
    barmode='stack',
    title='World Cup 2026 Semifinal Predictions — Model Win Probabilities',
    xaxis_title='Win Probability (%)',
    showlegend=False,
    height=350,
    width=800,
    font=dict(size=14),
    plot_bgcolor='white'
)

fig.write_image("semifinal_predictions.png", scale=2)
fig.show()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 19.4 MB/s eta 0:00:00:00:0100:01


In [19]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Bar(y=['France vs Spain', 'England vs Argentina'], x=[47.5, 38.1],
                      orientation='h', marker_color='#4C72B0',
                      text=['France 47.5%', 'England 38.1%'], textposition='inside'))
fig.add_trace(go.Bar(y=['France vs Spain', 'England vs Argentina'], x=[52.5, 61.9],
                      orientation='h', marker_color='#DD8452',
                      text=['Spain 52.5%', 'Argentina 61.9%'], textposition='inside'))
fig.update_layout(barmode='stack', title='World Cup 2026 Semifinal Predictions — Model Win Probabilities',
                   xaxis_title='Win Probability (%)', showlegend=False, height=350, width=800,
                   font=dict(size=14), plot_bgcolor='white')

fig.write_image("semifinal_predictions.png", scale=2)
fig.show()

## Save model + lookup files (for the Gradio app)

In [20]:
import joblib

# 1. Save the trained model
joblib.dump(log_reg, 'model.pkl')

# 2. Save current Elo ratings for every team
ratings_df = pd.DataFrame(list(ratings.items()), columns=['team', 'elo'])
ratings_df.to_csv('elo_ratings.csv', index=False)

# 3. Save each team's latest known form
form_df = latest_form.reset_index()[['team', 'form_gf', 'form_ga']]
form_df.to_csv('team_form.csv', index=False)

print("Saved: model.pkl, elo_ratings.csv, team_form.csv")

import os
print(os.listdir('.'))

Saved: model.pkl, elo_ratings.csv, team_form.csv
['semifinal_predictions.png', 'team_form.csv', 'elo_ratings.csv', 'model.pkl', '.virtual_documents']
